In [6]:
import pickle
import numpy as np
from sklearn.feature_extraction.text import CountVectorizer
import config as config
import green_tsetlin as gt
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split

In [4]:
path = "/home/bigtech/data/verbosius/imdb/testing/bad_texts/imdb_bad_x_1.pkl"
bad = pickle.load(open(path, "rb"))

In [25]:
bad["x"]

['i must admit that at the beginning i was sort of reticent about watching this movie i thought it was this stupid little romantic film about a french woman who meets in the train an american and decides to visit vienna with him i was not actually enchanted about this kind of script since it continued to make me believe that it is just a movie still i watched it and i was amazed before sunrise is one of the few films who dare to talk in a rather philosophical way wondering about the fact that in the moment of our birth we are sentenced to death or that it is a middling idea that fact that a couple should rest together for eternity or that we humans can afford sometimes to live in fairy tales the ending was wonderfully chosen we do not know if they will meet again in six months at six oclock in viennas station in our optimism we sincerely hope so the actors acted in a very good manner so that i began to believe that i myself could live a love story just like this',
 'very badly made fil

In [5]:
print(f"n texts : {len(bad['x'])}")
print(f"class diff : {np.mean(bad['y'])}")

n texts : 2998
class diff : 0.4839893262174783


In [8]:
train_x, eval_x, train_y, eval_y = train_test_split([x for x in bad["x"]],
                                                    np.array(bad["y"], dtype=np.uint32),
                                                    test_size=0.2,
                                                    random_state=42)

vectorizer = CountVectorizer(max_features=config.MAX_FEATURES*6,
                                 max_df=config.MAX_DF, 
                                 min_df=config.MIN_DF,
                                 ngram_range=config.N_GRAM_RANGE,
                                 binary=True,
                                 dtype=np.uint8,
                                 stop_words = config.STOPWORDS)
    
train_x_bin = vectorizer.fit_transform(train_x).toarray()
eval_x_bin = vectorizer.transform(eval_x).toarray()

0.8


In [18]:
logit = LogisticRegression(max_iter=1000, random_state=42)
logit.fit(train_x_bin, train_y)

print("eval acc :", logit.score(eval_x_bin, eval_y))

eval acc : 0.8


In [24]:
tm = gt.TsetlinMachine(n_literals=train_x_bin.shape[1], 
                           n_clauses=1000, 
                           n_classes=2,
                           s=config.S,
                           n_literal_budget=4, 
                           )

tm.set_train_data(train_x_bin, train_y)
    
tm.set_test_data(eval_x_bin, eval_y) 

trainer = gt.Trainer(1500, 
                        n_epochs=10, 
                        seed=42, 
                        n_jobs=5, 
                        early_exit_acc=0.86,
                        progress_bar=True)

trainer.train(tm)    

Processing epoch 10 of 10, train acc: 0.723, best test score: 0.763 (epoch: 6): 100%|██████████| 10/10 [00:10<00:00,  1.03s/it]


{'best_test_score': 0.7633333333333333,
 'best_test_epoch': 6,
 'n_epochs': 10,
 'train_log': [0.5596330275229358,
  0.6480400333611342,
  0.6755629691409508,
  0.6747289407839867,
  0.707256046705588,
  0.7043369474562136,
  0.7364470391993327,
  0.7297748123436196,
  0.7326939115929941,
  0.7226855713094246],
 'test_log': [0.6666666666666666,
  0.7116666666666667,
  0.7116666666666667,
  0.7533333333333333,
  0.7416666666666667,
  0.74,
  0.7633333333333333,
  0.7433333333333333,
  0.7633333333333333,
  0.7533333333333333],
 'did_early_exit': False}